# GRACE H3 — Bắt đầu full pipeline tại đây

Hướng mới: **CPU logistic gate → Platt calibration → constrained routing → GRACE cho inspect**. Mặc định full test Devign, skip-low; direct-high tắt. Gate-only và full GRACE là đối chứng. Không có GPU/Kaggle stage ở cấu hình H3 độc lập.

| Bước | Môi trường | Handoff |
|---|---|---|
| 1 | Local | Xuất cpu-inputs.zip; chuyển sang colab_cpu_prepare.ipynb |
| 2 | Colab CPU, accelerator None | Fit gate/calibration/thresholds, route, prepare GRACE; trả cpu-output.zip |
| 3 | Local | Import, preflight → Vertex → merge full cohort → paired evaluation |

Notebook dừng ở Bước1 khi thiếu CPU output. Tải output vào bundles/ rồi Run All lại để tiếp tục. full_reference=true thêm các query bị skip vào nhánh đối chứng; cascade chạy inference trước để số request của nó không bị che bởi prewarming cache.

Đọc docs/METHOD.md về ràng buộc lỗi, docs/RUNBOOK.md về cloud và H1+H3 tùy chọn.

In [1]:
from pathlib import Path
import os,sys,json,subprocess,zipfile
ROOT=Path('/home/quang-dung/.local/share/grace-h3/GRACE-H3-CalibratedCascade-recovered-r1')
os.environ["GRACE_H3_ROOT"]=str(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
INSTALL_DEPENDENCIES=False
PREPARE_ON_COLAB=False
RUN_INFERENCE=True  # Set True to make API calls.
LLM_PROMPT_PERCENT=50  # First N%; raise to 100 later to resume using cached successes.
OUTPUT_BUNDLE=ROOT/'bundles'/'grace-h3-cpu-output.zip'
H1_HEADS_BUNDLE=ROOT/'bundles'/'grace-h1-trained-heads.zip' # optional H1+H3 only
config=json.loads((ROOT/'configs'/'experiment.json').read_text(encoding='utf-8'))
config["datasets"] = ["reveal"]
config["split"] = "test"
config["full_reference"] = True
config["retrieval_arm"] = "baseline_whitening"

print(ROOT);print(json.dumps(config,indent=2))

/home/quang-dung/.local/share/grace-h3/GRACE-H3-CalibratedCascade-recovered-r1
{
  "bootstrap": 1000,
  "datasets": [
    "reveal"
  ],
  "full_reference": true,
  "gate": {
    "C": 1,
    "calibration_fraction": 0.5,
    "class_weight": null,
    "enable_direct_high": false,
    "family_alpha": 0.05,
    "high_grid": [
      0.7,
      0.8,
      0.9,
      0.95,
      0.975,
      0.99
    ],
    "low_grid": [
      0.01,
      0.025,
      0.05,
      0.075,
      0.1,
      0.15,
      0.2,
      0.25,
      0.3
    ],
    "max_high_false_positive_fraction": 0.01,
    "max_iter": 1000,
    "max_lost_positive_fraction": 0.01,
    "min_high_precision": 0.98,
    "seed": 42
  },
  "graph_workers": 4,
  "inference_workers": 4,
  "retrieval_arm": "baseline_whitening",
  "split": "test"
}


In [2]:
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements.txt')])
from grace_h1.io import read_json,file_hash,load_local_environment
from grace_h3.portable import create_bundle,import_bundle,identity
from grace_h3.workflow import prepare_all,selected_jobs,head_for,preflight_all,infer_all,evaluate_all
import pandas as pd
load_local_environment(ROOT)
sys.path.insert(0, str(ROOT / "tools"))
from prefix_inference import run_prefix_all, evaluate_cached_all


/home/quang-dung/.venvs/grace-h2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Bước 1 — Chuyển phần CPU/RAM sang Colab

Lần đầu: mang grace-h3-cpu-inputs.zip và grace-h3-joern-runtime.zip trong bundles/ lên MyDrive/GRACE-H3-transfer. Upload notebooks/colab_cpu_prepare.ipynb vào Colab, chọn accelerator **None**, Run All.

Colab dùng train để fit logistic, chia validation thành calibration và threshold selection riêng, khóa policy, rồi chỉ prepare GRACE cho inspect. Khi full_reference=true, Colab prepare thêm full cohort cho đối chứng. Không gọi LLM trên Colab.

Tải grace-h3-cpu-output.zip về bundles/ local, Run All notebook này lại. Có thể chọn PREPARE_ON_COLAB=False để thực hiện offline trên local với Joern.

**H1+H3 tùy chọn:** sửa retrieval_arm trong configs/experiment.json thành h1_supcon_hard trước khi xuất bundle; mang trained-heads.zip từ project H1 về H1_HEADS_BUNDLE. Nếu chưa train H1, chạy notebook Kaggle của project H1 trước. Hai nhánh reference/cascade đều dùng cùng H1 head; gate H3 vẫn dùng frozen CodeT5.
**Ubuntu dual boot:** nếu bước prepare lỗi Joern/GraphML, chạy notebook [prepare-cpu-ubuntu.ipynb](prepare-cpu-ubuntu.ipynb). Notebook này hướng dẫn copy/resume project tại ~/.local/share/grace-h3 và dùng Java/Joern từ ~/.cache/grace-h2 giống H2 Ubuntu. Kết quả chuẩn bị nằm ở bản Ubuntu; tiếp tục full-pipeline tại bản đó.


In [3]:
if config['retrieval_arm']!='baseline_whitening':
    if H1_HEADS_BUNDLE.exists(): print(import_bundle(ROOT,H1_HEADS_BUNDLE,'trained_heads'))
    for dataset in config['datasets']:
        try: head_for(ROOT,dataset,config['retrieval_arm'])
        except (FileNotFoundError,KeyError,ValueError) as exc:
            raise RuntimeError('H1+H3 cần trained-heads.zip matching từ project H1; đặt tại H1_HEADS_BUNDLE. H3 độc lập dùng retrieval_arm=baseline_whitening.') from exc
if PREPARE_ON_COLAB:
    if OUTPUT_BUNDLE.exists(): print(import_bundle(ROOT,OUTPUT_BUNDLE,'cpu_output'))
    try:
        jobs=selected_jobs(ROOT,config)
    except (FileNotFoundError,KeyError,ValueError) as exc:
        bundle=ROOT/'bundles'/'grace-h3-cpu-inputs.zip';ready=False
        if bundle.exists() and config['retrieval_arm']=='baseline_whitening':
            with zipfile.ZipFile(bundle) as z:
                m=json.loads(z.read('bundle_manifest.json'))
                ready=m.get('schema')=='grace_h3_bundle_v1' and m.get('kind')=='cpu_inputs' and m['job']=={'source':identity(ROOT),'data_manifest':file_hash(ROOT/'data'/'manifest.json'),'config':config}
        print({'reuse_existing_inputs':ready,'path':str(bundle)} if ready else create_bundle(ROOT,bundle,'cpu_inputs'))
        raise RuntimeError('DỪNG: chạy notebooks/colab_cpu_prepare.ipynb trên Colab CPU với cpu-inputs.zip và joern-runtime.zip; tải cpu-output.zip vào bundles/ rồi Run All lại. '+str(exc)) from exc
else:
    prepare_all(ROOT,config);jobs=selected_jobs(ROOT,config)
print('Prepared jobs:',[j['dataset'] for _,j in jobs])

Prepared jobs: ['reveal']


## Bước 2 — Đọc policy đã khóa

low=0 nghĩa là chưa có ngưỡng skip đạt constraint; tất cả đi inspect. high=null nghĩa là direct-high chưa bật/không đạt constraint. Bound Clopper–Pearson/Bonferroni chỉ có ý nghĩa nominal trên assumptions đã ghi; không bảo đảm recall test hoặc domain mới. Không đổi thresholds sau khi xem test.

In [4]:
policies=[]
for folder,job in jobs:
    g=read_json(folder/'gate_manifest.json')
    policies.append({'dataset':job['dataset'],'low':g['policy']['low'],'high':g['policy']['high'],
                     'branches':job['branch_counts'],'selection_n':g['counts']['selection'],
                     'selection_brier':g['selection_calibrated']['brier'],'fit_seconds':g['fit_seconds']})
display(pd.DataFrame(policies))

,dataset,low,high,branches,selection_n,selection_brier,fit_seconds
0,reveal,0.0,None,{'inspect': 2274},1098,0.067218,7.831127


## Bước 3 — Preflight rồi inference local

RUN_INFERENCE=True gọi Vertex cho exact prompts còn thiếu. Cascade gọi trước; full reference sau có thể dùng lại response inspect. full_reference=true có thể khiến cả benchmark vẫn gọi LLM cho toàn bộ cohort, dù deployed cascade chỉ cần tỷ lệ inspect. Báo cáo tách chi phí reference, routing và cache.

ADC/Vertex dùng môi trường local; không đưa credential lên Colab. Parser invalid không bị ép thành safe.

In [5]:
plans=run_prefix_all(ROOT,config,percent=LLM_PROMPT_PERCENT,allow_api=False)
for p in plans: print(json.dumps(p,indent=2))

reveal: first 1137/2274 prompts (50%); already successful across full set: 0/2274
{
  "dataset": "reveal",
  "status": "partial_or_planned",
  "prompt_percent": 50,
  "cascade": {
    "queries": 1137,
    "unique_prompts": 1137,
    "cached_queries": 0,
    "new_unique_requests": 1137,
    "imported_prior_responses": 0,
    "request_spec": {
      "model": "DeepSeek-V4-Flash",
      "provider": "FPT AI Factory",
      "base_url": "https://token-api.fpt.ai/v1",
      "inference": {
        "temperature": 0.0,
        "seed": 42,
        "max_tokens": 256,
        "retry_policy": "max_attempts=5; exponential_backoff=5s..120s; exhausted_request=invalid",
        "parser": "../Grace-with-filters-v2/[CONTRACT]/parser/parse_prediction.py::parse_prediction",
        "parser_contract": "exact_last_line_binary_v1"
      }
    },
    "worst_case_attempts": 5685,
    "status": "prepared_cache_only",
    "action": "run again with allow_api=True to call FPT for missing exact prompts",
    "prompt_p

In [6]:
reports=run_prefix_all(ROOT,config,percent=LLM_PROMPT_PERCENT,allow_api=RUN_INFERENCE)
for report in reports: print(json.dumps(report,indent=2))

reveal: first 1137/2274 prompts (50%); already successful across full set: 0/2274
LLM unique requests 25/1137
LLM unique requests 50/1137
LLM unique requests 75/1137
LLM unique requests 100/1137
LLM unique requests 125/1137
LLM unique requests 150/1137
LLM unique requests 175/1137
LLM unique requests 200/1137
LLM unique requests 225/1137
LLM unique requests 250/1137
LLM unique requests 275/1137
LLM unique requests 300/1137
LLM unique requests 325/1137
LLM unique requests 350/1137
LLM unique requests 375/1137
LLM unique requests 400/1137
LLM unique requests 425/1137
LLM unique requests 450/1137
LLM unique requests 475/1137
LLM unique requests 500/1137
LLM unique requests 525/1137
LLM unique requests 550/1137
LLM unique requests 575/1137
LLM unique requests 600/1137
LLM unique requests 625/1137
LLM unique requests 650/1137
LLM unique requests 675/1137
LLM unique requests 700/1137
LLM unique requests 725/1137
LLM unique requests 750/1137
LLM unique requests 775/1137
LLM unique requests 80

## Bước 4 — Đánh giá toàn cohort

Có cascade, gate-only, full reference, branch FN/FP, invalid rate, recall_all_positives và paired CI. ROC-AUC/AP chỉ thuộc gate score, không gắn tên AUC của final cascade. Các metric F1/MCC condition trên valid; invalid được giữ trong báo cáo full cohort.

In [ ]:
# Step 4 — evaluate every successful response already available, even before 100%.
# Reload the helper so this cell can be run immediately in the existing kernel.
import importlib, prefix_inference
importlib.reload(prefix_inference)
summary=prefix_inference.evaluate_cached_all(ROOT,config)
rows=[]
for result in summary['datasets']:
    for arm in ('cascade','gate_only','full_reference'):
        if arm not in result:
            continue
        metric=result[arm]
        coverage=result['coverage']['cascade' if arm=='gate_only' else arm]
        rows.append({'dataset':result['dataset'],'arm':arm,
                     'inferred_n':metric['n'],'total_prompts':coverage['total_prompts'],
                     'inferred_percent':coverage['inferred_percent'],
                     'pending_n':coverage['pending_n'],
                     'f1_valid':metric['f1'],'mcc_valid':metric['mcc'],
                     'accuracy_valid':metric['accuracy_valid_only'],
                     'precision':metric['precision'],'recall':metric['recall'],
                     'invalid_rate':metric['invalid_rate']})
    print(result['dataset'], 'Status:',result['status'], 'Coverage:',result['coverage'])
display(pd.DataFrame(rows))
print('Status:',summary['status'])
print('Chưa inference xong; metrics chỉ trên phần đã có phản hồi thành công.'
      if summary['status']=='partial' else 'Đã inference toàn bộ prompt.')
print('Saved:',ROOT/'runs/partial_experiment_summary.json')
